# 🌾 GraminRoute — ML Training Notebook
**Version 2.0 · Jangaon District, Telangana**

| Stage | Model | Purpose |
|-------|-------|---------|
| 1 | XGBoost Risk Model | Per-shop risk from 9 features |
| 2 | SpatialGNN (GATv2) | Road-network spatial propagation |
| 3 | XGBoost Recommender | Ranked distributor selection |

**Data:** 500 real Jangaon shops derived from DataCo Global Supply Chain dataset (180k real delivery orders).  
Run all cells top-to-bottom. Models save to `backend/models/` automatically.


In [ ]:
# ── Colab Setup ────────────────────────────────────────────────────────────────
import sys, os

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    if not os.path.exists('/content/Gramin-Route'):
        os.system('git clone https://github.com/Atharva-sp21/Gramin-Route.git /content/Gramin-Route')
    os.chdir('/content/Gramin-Route/notebook')
    os.system('pip install -q xgboost scikit-learn shap networkx torch torch-geometric')
    print(f"Working dir: {os.getcwd()}")
else:
    print("Running locally — no setup needed")


## 0 · Setup & Imports

In [ ]:
import sys, os, warnings, pickle
from pathlib import Path
from datetime import date, timedelta

NOTEBOOK_DIR = Path().resolve()
REPO_ROOT    = NOTEBOOK_DIR.parent
BACKEND_DIR  = REPO_ROOT / "backend"
DATA_DIR     = REPO_ROOT / "data"
MODELS_DIR   = BACKEND_DIR / "models"

sys.path.insert(0, str(BACKEND_DIR))
MODELS_DIR.mkdir(parents=True, exist_ok=True)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import (roc_curve, auc, confusion_matrix,
                             classification_report, ConfusionMatrixDisplay)
import xgboost as xgb
import shap

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({
    'figure.dpi': 110,
    'font.family': 'DejaVu Sans',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
})
RISK_CMAP = plt.cm.RdYlGn_r

print("Imports ready")
print(f"  Data dir   : {DATA_DIR}")
print(f"  Models dir : {MODELS_DIR}")


## 1 · Data Loading & Exploratory Analysis

**Source:** 500 Jangaon District shops derived from the DataCo Global Supply Chain dataset (180,519 real delivery orders).  
Real delivery outcomes → `Late_delivery_risk` is a genuine label, not a synthetic rule.


In [ ]:
df = pd.read_csv(DATA_DIR / "jangaon_shops.csv")
print(f"Shape: {df.shape}")
print(f"High-risk shops (risk >= 0.5): {(df['Late_delivery_risk'] >= 0.5).sum()} / {len(df)}")
df.head(8)


In [ ]:
df.describe().round(2)


### 1.1 Feature Distributions

In [ ]:
numeric_cols = ['Stock', 'Sales', 'Days', 'Profit_Margin', 'Shelf_Life', 'Credit_Score']
col_labels   = ['Stock (units)', 'Daily Sales', 'Lead Time (days)',
                'Profit Margin (%)', 'Shelf Life (days)', 'Credit Score']

fig, axes = plt.subplots(2, 3, figsize=(14, 7))
axes = axes.flatten()
colors = sns.color_palette("husl", 6)

for i, (col, label) in enumerate(zip(numeric_cols, col_labels)):
    ax = axes[i]
    ax.hist(df[col], bins=25, color=colors[i], edgecolor='white', alpha=0.85)
    ax.set_title(label)
    ax.set_ylabel('Count')
    mean_val = df[col].mean()
    ax.axvline(mean_val, color='black', linestyle='--', linewidth=1.2,
               label=f'Mean: {mean_val:.1f}')
    ax.legend(fontsize=9)

fig.suptitle('Feature Distributions — 500 Jangaon Shops (DataCo-derived)',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(MODELS_DIR / 'eda_distributions.png', bbox_inches='tight')
plt.show()


### 1.2 Correlation Heatmap

In [ ]:
corr_cols = numeric_cols + ['Late_delivery_risk']
corr = df[corr_cols].corr()

fig, ax = plt.subplots(figsize=(9, 7))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            mask=mask, linewidths=0.5, ax=ax, annot_kws={'size': 9},
            cbar_kws={'shrink': 0.8})
ax.set_title('Feature Correlation Matrix — Jangaon Shops', fontsize=14,
             fontweight='bold', pad=12)
plt.tight_layout()
plt.savefig(MODELS_DIR / 'eda_correlation.png', bbox_inches='tight')
plt.show()


### 1.3 Risk Label Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(df['Late_delivery_risk'], bins=15, color='#e67e22',
             edgecolor='white', alpha=0.85)
axes[0].set_title('Late Delivery Risk — Continuous (DataCo real data)')
axes[0].set_xlabel('Risk Score')
axes[0].set_ylabel('Shop Count')

binary_labels = (df['Late_delivery_risk'] >= 0.5).astype(int)
pie_data = binary_labels.value_counts()
axes[1].pie(pie_data.values, labels=['Low Risk', 'High Risk'],
            colors=['#27ae60', '#e74c3c'], autopct='%1.1f%%', startangle=90,
            textprops={'fontsize': 11}, pctdistance=0.75,
            wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[1].set_title('Binarised Labels (threshold >= 0.5)')

plt.suptitle('Late Delivery Risk — 500 Jangaon Shops', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(MODELS_DIR / 'eda_risk_labels.png', bbox_inches='tight')
plt.show()
print(f"High-risk: {binary_labels.sum()} / {len(binary_labels)} ({binary_labels.mean()*100:.1f}%)")


### 1.4 Village Map — Risk by Location

In [ ]:
fig, ax = plt.subplots(figsize=(11, 8))

sc = ax.scatter(df['Longitude'], df['Latitude'],
                c=df['Late_delivery_risk'], cmap=RISK_CMAP,
                s=60, edgecolors='white', linewidths=0.4, alpha=0.80, zorder=3)

high_risk = df.nlargest(12, 'Late_delivery_risk')
for _, row in high_risk.iterrows():
    ax.annotate(row['Village_Name'], (row['Longitude'], row['Latitude']),
                fontsize=7, ha='left', va='bottom',
                xytext=(3, 3), textcoords='offset points', color='#c0392b')

cbar = plt.colorbar(sc, ax=ax, shrink=0.7, pad=0.02)
cbar.set_label('Late Delivery Risk', fontsize=11)
ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude')
ax.set_title('Jangaon District — 500 Shop Locations\nColoured by real delivery risk (DataCo)',
             fontsize=13, fontweight='bold')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(MODELS_DIR / 'eda_village_map.png', bbox_inches='tight')
plt.show()


## 2 · Feature Engineering — Festival Context

Products are already assigned per shop in `jangaon_shops.csv`.  
We derive 3 festival features at runtime from the Indian festival calendar.


In [ ]:
from ml.festival_calendar import get_festival_context, FESTIVAL_CALENDAR

print("Festival Calendar — Jangaon District")
print("=" * 60)
for name, info in FESTIVAL_CALENDAR.items():
    print(f"  {name:<12} | Month {info['month']:>2} Day {info['day']:>2} | "
          f"Spike x{info['spike_factor']:.1f} | Prep {info['prep_days']} days")
    print(f"             | Products: {', '.join(info['products'])}")


In [ ]:
TODAY = date.today()

festival_feats = df['Product_Name'].apply(
    lambda p: get_festival_context(p, TODAY)
)

df['days_to_festival_raw']  = [f['days_to_festival'] for f in festival_feats]
df['spike_factor']          = [f['spike_factor']     for f in festival_feats]
df['product_affinity']      = [f['product_affinity'] for f in festival_feats]
df['festival_name']         = [f['festival_name']    for f in festival_feats]
df['in_prep_window']        = [f['in_prep_window']   for f in festival_feats]
df['days_to_festival_norm'] = (df['days_to_festival_raw'] / 30.0).clip(0, 1)
df['spike_factor_norm']     = df['spike_factor'] / 3.0

print("Festival features added:")
print(df[['Village_Name', 'Product_Name', 'festival_name',
          'days_to_festival_raw', 'spike_factor', 'product_affinity']].head(10).to_string(index=False))


### 2.1 Festival Feature Visualisation

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

festival_spike = {k: v['spike_factor'] for k, v in FESTIVAL_CALENDAR.items()}
bars = axes[0].barh(list(festival_spike.keys()), list(festival_spike.values()),
                    color=sns.color_palette('Oranges_r', len(festival_spike)),
                    edgecolor='white')
axes[0].set_xlabel('Demand Spike Factor (x baseline)')
axes[0].set_title('Demand Spike by Festival')
for bar, val in zip(bars, festival_spike.values()):
    axes[0].text(val + 0.02, bar.get_y() + bar.get_height()/2,
                 f'x{val}', va='center', fontsize=10, fontweight='bold')
axes[0].set_xlim(0, 3.5)
axes[0].axvline(1.0, color='grey', linestyle='--', linewidth=1, label='Baseline')
axes[0].legend()

affinity_counts = df['product_affinity'].value_counts()
axes[1].bar(['Not Affected', 'Festival Product'],
            [affinity_counts.get(0.0, 0), affinity_counts.get(1.0, 0)],
            color=['#3498db', '#e67e22'], edgecolor='white', width=0.5)
axes[1].set_ylabel('Number of Shops')
axes[1].set_title('Product Affinity Distribution')
for i, v in enumerate([affinity_counts.get(0.0, 0), affinity_counts.get(1.0, 0)]):
    axes[1].text(i, v + 1, str(v), ha='center', fontsize=11, fontweight='bold')

axes[2].hist(df['days_to_festival_raw'], bins=15, color='#9b59b6',
             edgecolor='white', alpha=0.85)
axes[2].set_xlabel('Days to Next Festival')
axes[2].set_ylabel('Count')
axes[2].set_title('Days to Next Festival (per product)')
axes[2].axvline(df['days_to_festival_raw'].mean(), color='black',
                linestyle='--', linewidth=1.5,
                label=f"Mean: {df['days_to_festival_raw'].mean():.0f}d")
axes[2].legend()

plt.suptitle('Festival Feature Engineering', fontsize=14, fontweight='bold', y=1.03)
plt.tight_layout()
plt.savefig(MODELS_DIR / 'feat_eng_festival.png', bbox_inches='tight')
plt.show()


## 3 · XGBoost Risk Model

Trained on **500 real shops** from DataCo supply chain data — no synthetic augmentation needed.  
`Late_delivery_risk` labels come from actual delivery outcomes aggregated per location.


In [ ]:
FEATURE_NAMES = [
    "stock_ratio", "sales_ratio", "lead_time_ratio",
    "margin_ratio", "shelf_life_ratio", "credit_ratio",
    "days_to_festival", "spike_factor", "product_affinity",
]

X = np.column_stack([
    df['Stock'].values          / 200.0,
    df['Sales'].values          / 50.0,
    df['Days'].values           / 7.0,
    df['Profit_Margin'].values  / 60.0,
    df['Shelf_Life'].values     / 365.0,
    df['Credit_Score'].values   / 900.0,
    df['days_to_festival_norm'].values,
    df['spike_factor_norm'].values,
    df['product_affinity'].values,
]).astype(np.float32)

y = (df['Late_delivery_risk'].values >= 0.5).astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Feature matrix : {X.shape}")
print(f"Train : {X_train.shape}  |  Test: {X_test.shape}")
print(f"High-risk in train: {y_train.mean()*100:.1f}%")
print("Training on 100% real DataCo delivery data")


In [ ]:
xgb_risk = xgb.XGBClassifier(
    n_estimators=200, max_depth=6, learning_rate=0.1,
    subsample=0.8, colsample_bytree=0.8,
    eval_metric="logloss", random_state=42,
    early_stopping_rounds=20,
)
xgb_risk.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)

results    = xgb_risk.evals_result()
train_loss = results['validation_0']['logloss']

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(train_loss, color='#e74c3c', linewidth=2, label='Validation log-loss')
ax.axhline(min(train_loss), color='grey', linestyle='--', linewidth=1,
           label=f'Best: {min(train_loss):.4f} @ iter {train_loss.index(min(train_loss))}')
ax.set_xlabel('Boosting Round')
ax.set_ylabel('Log-loss')
ax.set_title('XGBoost Risk Model — Training Curve', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig(MODELS_DIR / 'risk_training_curve.png', bbox_inches='tight')
plt.show()


In [ ]:
y_prob = xgb_risk.predict_proba(X_test)[:, 1]
y_pred = xgb_risk.predict(X_test)
fpr, tpr, _ = roc_curve(y_test, y_prob)
roc_auc = auc(fpr, tpr)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].plot(fpr, tpr, color='#e74c3c', linewidth=2.5, label=f'AUC = {roc_auc:.3f}')
axes[0].plot([0, 1], [0, 1], 'k--', linewidth=1, alpha=0.5)
axes[0].fill_between(fpr, tpr, alpha=0.08, color='#e74c3c')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curve — XGBoost Risk Model', fontweight='bold')
axes[0].legend(loc='lower right', fontsize=12)
axes[0].set_xlim([0, 1]); axes[0].set_ylim([0, 1.01])

cm = confusion_matrix(y_test, y_pred)
ConfusionMatrixDisplay(cm, display_labels=['Low Risk', 'High Risk']).plot(
    ax=axes[1], cmap='Blues', colorbar=False)
axes[1].set_title('Confusion Matrix', fontweight='bold')
axes[1].grid(False)

plt.tight_layout()
plt.savefig(MODELS_DIR / 'risk_roc_cm.png', bbox_inches='tight')
plt.show()

print(f"AUC: {roc_auc:.4f}")
print(classification_report(y_test, y_pred, target_names=['Low Risk', 'High Risk']))


### 3.1 SHAP Feature Importance

In [ ]:
explainer   = shap.TreeExplainer(xgb_risk)
shap_values = explainer.shap_values(X_test)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

plt.sca(axes[0])
shap.summary_plot(shap_values, X_test, feature_names=FEATURE_NAMES,
                  plot_type='dot', show=False)
axes[0].set_title('SHAP Beeswarm — Impact Distribution', fontweight='bold', pad=10)

plt.sca(axes[1])
shap.summary_plot(shap_values, X_test, feature_names=FEATURE_NAMES,
                  plot_type='bar', show=False)
axes[1].set_title('SHAP Bar — Mean |Impact| per Feature', fontweight='bold', pad=10)

plt.savefig(MODELS_DIR / 'risk_shap.png', bbox_inches='tight', dpi=120)
plt.show()


In [ ]:
shop_idx = np.argmax(y_prob)

shap_ex = shap.Explanation(
    values        = shap_values[shop_idx],
    base_values   = explainer.expected_value,
    data          = X_test[shop_idx],
    feature_names = FEATURE_NAMES,
)
fig, ax = plt.subplots(figsize=(10, 5))
shap.waterfall_plot(shap_ex, show=False)
plt.title(f'SHAP Waterfall — Highest-Risk Shop\nPredicted Risk: {y_prob[shop_idx]:.2%}',
          fontweight='bold', pad=12)
plt.tight_layout()
plt.savefig(MODELS_DIR / 'risk_shap_waterfall.png', bbox_inches='tight')
plt.show()


In [ ]:
risk_model_path = MODELS_DIR / "xgb_risk_model.pkl"
with open(risk_model_path, "wb") as f:
    pickle.dump(xgb_risk, f)
print(f"XGBoost Risk Model saved -> {risk_model_path}")


## 4 · Village Road Graph Construction

500 shops as nodes, connected to their 5 nearest neighbours.  
Edge weights represent road type (highway=1.0, state road=0.6, rural=0.3).


In [ ]:
import networkx as nx
from math import radians, sin, cos, sqrt, atan2
from sklearn.neighbors import NearestNeighbors
from collections import Counter

G    = nx.Graph()
lats = df['Latitude'].values
lons = df['Longitude'].values
risks = (df['Late_delivery_risk'].values >= 0.5).astype(float)

for i in range(len(df)):
    G.add_node(i, lat=lats[i], lon=lons[i], risk=risks[i],
               village=df['Village_Name'].iloc[i])

K = 5
coords_rad = np.radians([[lats[i], lons[i]] for i in range(len(df))])
nbrs = NearestNeighbors(n_neighbors=K+1, metric='haversine').fit(coords_rad)
distances, indices = nbrs.kneighbors(coords_rad)

for i in range(len(df)):
    for j_pos in range(1, K+1):
        j    = indices[i][j_pos]
        dist = distances[i][j_pos] * 6371.0
        if not G.has_edge(i, j):
            if dist < 3:
                wt, rt = 1.0, 'highway'
            elif dist < 8:
                wt, rt = 0.6, 'state'
            else:
                wt, rt = 0.3, 'rural'
            G.add_edge(i, j, weight=wt, road_type=rt, dist_km=round(dist, 2))

road_types = Counter(nx.get_edge_attributes(G, 'road_type').values())
print(f"Graph : {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")
print(f"Avg degree : {sum(dict(G.degree()).values()) / G.number_of_nodes():.1f}")
print(f"Components : {nx.number_connected_components(G)}")
print(f"Road types : {dict(road_types)}")


In [ ]:
from matplotlib.lines import Line2D

fig, ax = plt.subplots(figsize=(13, 9))
pos = {i: (lons[i], lats[i]) for i in range(len(df))}

edge_colors, edge_widths = [], []
for u, v, data in G.edges(data=True):
    rt = data['road_type']
    if rt == 'highway':
        edge_colors.append('#2c3e50'); edge_widths.append(2.0)
    elif rt == 'state':
        edge_colors.append('#7f8c8d'); edge_widths.append(1.3)
    else:
        edge_colors.append('#bdc3c7'); edge_widths.append(0.7)

node_colors = [RISK_CMAP(risks[i]) for i in range(len(df))]
node_sizes  = [40 + 120 * risks[i] for i in range(len(df))]

nx.draw_networkx_edges(G, pos, ax=ax, edge_color=edge_colors,
                       width=edge_widths, alpha=0.55)
nx.draw_networkx_nodes(G, pos, ax=ax, node_color=node_colors,
                       node_size=node_sizes, edgecolors='white', linewidths=0.5)

high_nodes = {i: df['Village_Name'].iloc[i][:10] for i in range(len(df)) if risks[i] == 1}
if len(high_nodes) > 15:
    high_nodes = dict(list(high_nodes.items())[:15])
nx.draw_networkx_labels(G, pos, labels=high_nodes, ax=ax,
                        font_size=6, font_color='#c0392b')

legend_elements = [
    Line2D([0], [0], color='#2c3e50', linewidth=2,   label='Highway (w=1.0)'),
    Line2D([0], [0], color='#7f8c8d', linewidth=1.3, label='State Road (w=0.6)'),
    Line2D([0], [0], color='#bdc3c7', linewidth=0.7, label='Rural Road (w=0.3)'),
]
ax.legend(handles=legend_elements, loc='lower right', fontsize=9, framealpha=0.9)
ax.set_title('Jangaon Village Road Network — 500 Shops\nNode colour = risk level',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude')
ax.tick_params(left=True, bottom=True, labelleft=True, labelbottom=True)
plt.tight_layout()
plt.savefig(MODELS_DIR / 'village_graph.png', bbox_inches='tight', dpi=120)
plt.show()


## 5 · SpatialGNN Training (GATv2)

GATv2 propagates risk through the road network. Shops near many high-risk neighbours get elevated spatial risk scores.


In [ ]:
try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    from torch_geometric.data import Data
    from torch_geometric.nn import GATv2Conv
    TORCH_OK = True
    print(f"PyTorch {torch.__version__} | PyG ready")
except ImportError as e:
    TORCH_OK = False
    print(f"PyTorch/PyG not found: {e}")
    print("Skipping GNN — XGBoost risk scores will be used directly.")


In [ ]:
if TORCH_OK:
    from ml.model_def import SpatialGNN

    xgb_risk_scores = xgb_risk.predict_proba(X)[:, 1].astype(np.float32)
    X_10 = np.column_stack([X, xgb_risk_scores])

    node_x = torch.tensor(X_10, dtype=torch.float32)
    node_y = torch.tensor(risks, dtype=torch.float32).unsqueeze(1)

    edge_list, edge_weights = [], []
    for u, v, data in G.edges(data=True):
        edge_list.extend([[u, v], [v, u]])
        edge_weights.extend([data['weight'], data['weight']])

    edge_index = torch.tensor(edge_list, dtype=torch.long).t().contiguous()
    edge_attr  = torch.tensor(edge_weights, dtype=torch.float32).unsqueeze(1)

    pyg_data = Data(x=node_x, edge_index=edge_index, edge_attr=edge_attr, y=node_y)
    print(f"PyG Data: {pyg_data}")


In [ ]:
if TORCH_OK:
    model     = SpatialGNN(in_dim=10, hidden_dim=64)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.005, weight_decay=1e-4)
    criterion = nn.BCELoss()

    n_nodes   = pyg_data.num_nodes
    idx_all   = torch.randperm(n_nodes)
    train_idx = idx_all[:int(0.8 * n_nodes)]
    val_idx   = idx_all[int(0.8 * n_nodes):]

    train_losses, val_losses, val_accs = [], [], []

    model.train()
    for epoch in range(1, 201):
        optimizer.zero_grad()
        out        = model(pyg_data)
        train_loss = criterion(out[train_idx], pyg_data.y[train_idx])
        train_loss.backward()
        optimizer.step()

        model.eval()
        with torch.no_grad():
            val_out  = model(pyg_data)
            val_loss = criterion(val_out[val_idx], pyg_data.y[val_idx]).item()
            val_acc  = (((val_out[val_idx] > 0.5).float()) == pyg_data.y[val_idx]).float().mean().item()
        model.train()

        train_losses.append(train_loss.item())
        val_losses.append(val_loss)
        val_accs.append(val_acc)

        if epoch % 50 == 0:
            print(f"Epoch {epoch:>3} | Train Loss: {train_loss.item():.4f} "
                  f"| Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.3f}")

    print(f"\nBest val accuracy: {max(val_accs):.3f} @ epoch {val_accs.index(max(val_accs)) + 1}")


In [ ]:
if TORCH_OK:
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    epochs = range(1, 201)

    axes[0].plot(epochs, train_losses, color='#e74c3c', linewidth=1.8, label='Train Loss')
    axes[0].plot(epochs, val_losses,   color='#3498db', linewidth=1.8,
                 linestyle='--', label='Val Loss')
    axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('BCE Loss')
    axes[0].set_title('SpatialGNN — Training & Validation Loss', fontweight='bold')
    axes[0].legend()

    axes[1].plot(epochs, val_accs, color='#27ae60', linewidth=1.8)
    axes[1].axhline(max(val_accs), color='grey', linestyle='--', linewidth=1,
                    label=f'Peak: {max(val_accs):.3f}')
    axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy')
    axes[1].set_title('SpatialGNN — Validation Accuracy', fontweight='bold')
    axes[1].set_ylim([0, 1.05]); axes[1].legend()

    plt.tight_layout()
    plt.savefig(MODELS_DIR / 'gnn_training_curves.png', bbox_inches='tight')
    plt.show()


In [ ]:
if TORCH_OK:
    model.eval()
    with torch.no_grad():
        gnn_risk_scores = model(pyg_data).squeeze().numpy()

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    axes[0].hist(xgb_risk_scores, bins=25, color='#e74c3c', edgecolor='white', alpha=0.8)
    axes[0].set_title('XGBoost Risk\n(per-shop, no graph)'); axes[0].set_xlabel('Risk Score')

    axes[1].hist(gnn_risk_scores, bins=25, color='#3498db', edgecolor='white', alpha=0.8)
    axes[1].set_title('SpatialGNN Risk\n(road-network aware)'); axes[1].set_xlabel('Risk Score')

    sc = axes[2].scatter(xgb_risk_scores, gnn_risk_scores, c=risks,
                         cmap='RdYlGn_r', s=30, edgecolors='white',
                         linewidths=0.4, alpha=0.75)
    axes[2].plot([0, 1], [0, 1], 'k--', alpha=0.4, linewidth=1)
    plt.colorbar(sc, ax=axes[2], label='Ground Truth Risk')
    axes[2].set_xlabel('XGBoost Risk'); axes[2].set_ylabel('GNN Spatial Risk')
    axes[2].set_title('XGBoost vs SpatialGNN')

    plt.suptitle('Risk Score Comparison', fontsize=13, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig(MODELS_DIR / 'gnn_risk_comparison.png', bbox_inches='tight')
    plt.show()


In [ ]:
if TORCH_OK:
    gnn_path = MODELS_DIR / "spatial_gnn.pth"
    torch.save(model.state_dict(), gnn_path)
    print(f"SpatialGNN saved -> {gnn_path}")
else:
    print("Skipped GNN. API will use XGBoost risk directly.")


## 6 · XGBoost Distributor Recommender

Contextual Bandit framing: context = retailer state, arms = 3 distributors.  
Trained on synthetic context data (no real outcome data yet).  
When real delivery feedback accumulates, swap for LinUCB or Thompson Sampling.


In [ ]:
np.random.seed(42)
N_REC = 4000

r_risk     = np.random.uniform(0, 1, N_REC)
r_stockout = np.random.uniform(0.1, 1.0, N_REC)
r_festival = np.random.uniform(0, 1, N_REC)
r_credit   = np.random.uniform(0.4, 1.0, N_REC)
r_qty      = np.random.uniform(0, 1, N_REC)

X_rec = np.column_stack([r_risk, r_stockout, r_festival, r_credit, r_qty])
y_rec = np.where(r_risk > 0.7, 0,
        np.where(r_stockout < 0.17, 0,
        np.where(r_risk > 0.4, 1, 2)))

X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X_rec, y_rec, test_size=0.2, random_state=42, stratify=y_rec)

xgb_rec = xgb.XGBClassifier(
    n_estimators=150, max_depth=5, learning_rate=0.1,
    eval_metric="mlogloss", random_state=42)
xgb_rec.fit(X_train_r, y_train_r)

print(classification_report(y_test_r, xgb_rec.predict(X_test_r),
      target_names=['FastTrack', 'GraminRoute Hub', 'Budget Movers']))


In [ ]:
probas = xgb_rec.predict_proba(X_test_r)
DIST_NAMES  = ['FastTrack Logistics', 'GraminRoute Hub', 'Budget Movers']
DIST_COLORS = ['#e74c3c', '#3498db', '#27ae60']

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for i, (name, color) in enumerate(zip(DIST_NAMES, DIST_COLORS)):
    axes[i].hist(probas[:, i], bins=25, color=color, edgecolor='white', alpha=0.85)
    axes[i].set_title(f'{name}', fontweight='bold')
    axes[i].set_xlabel('Predicted Probability')
    axes[i].set_ylabel('Count' if i == 0 else '')
    mean_conf = probas[:, i].mean()
    axes[i].axvline(mean_conf, color='black', linestyle='--', linewidth=1.5,
                    label=f'Mean: {mean_conf:.2f}')
    axes[i].legend()

plt.suptitle('Distributor Recommendation — Confidence Distributions',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(MODELS_DIR / 'recommender_confidence.png', bbox_inches='tight')
plt.show()


In [ ]:
rec_path = MODELS_DIR / "xgb_recommender.pkl"
with open(rec_path, "wb") as f:
    pickle.dump(xgb_rec, f)
print(f"XGBoost Recommender saved -> {rec_path}")


## 7 · End-to-End Demo — Ramesh's Shop

A real Jangaon shop run through the full 5-stage pipeline.


In [ ]:
from ml.festival_calendar import get_festival_context
from ml.festival_predictor import compute_stockout_forecast

RAMESH = dict(
    shop_id='GHPR-042', village='Ghanpur',
    current_stock=18, daily_sales=8, lead_time_days=3,
    profit_margin=22.0, shelf_life=365, credit_score=720,
    product_name='Rice (50kg)',
)

print("=" * 56)
print("  RAMESH'S KIRANA SHOP — GraminRoute Pipeline")
print("=" * 56)
for k, v in RAMESH.items():
    print(f"  {k:<18}: {v}")


In [ ]:
print("\nSTAGE 1 — Feature Engineering")
festival_ctx = get_festival_context(RAMESH['product_name'])
print(f"  Festival : {festival_ctx['festival_name']} ({festival_ctx['days_to_festival']} days away)")
print(f"  Spike    : x{festival_ctx['spike_factor']}")
print(f"  Affected : {'YES' if festival_ctx['product_affinity'] == 1 else 'No'}")

features_9 = np.array([
    RAMESH['current_stock']  / 200.0,
    RAMESH['daily_sales']    / 50.0,
    RAMESH['lead_time_days'] / 7.0,
    RAMESH['profit_margin']  / 60.0,
    RAMESH['shelf_life']     / 365.0,
    RAMESH['credit_score']   / 900.0,
    min(festival_ctx['days_to_festival'] / 30.0, 1.0),
    festival_ctx['spike_factor'] / 3.0,
    festival_ctx['product_affinity'],
], dtype=np.float32)

print("\nSTAGE 2 — XGBoost Risk Score")
xgb_score = float(xgb_risk.predict_proba(features_9.reshape(1, -1))[0][1])
status = "CRITICAL" if xgb_score > 0.7 else "WARNING" if xgb_score > 0.4 else "STABLE"
print(f"  XGBoost risk : {xgb_score:.3f}  [{status}]")

shap_single = explainer.shap_values(features_9.reshape(1, -1))[0]
top3 = sorted(zip(FEATURE_NAMES, shap_single), key=lambda x: abs(x[1]), reverse=True)[:3]
print("  Top 3 SHAP drivers:")
for name, impact in top3:
    print(f"    {'up' if impact > 0 else 'dn'}  {name:<25} {impact:+.4f}")


In [ ]:
print("\nSTAGE 3 — SpatialGNN")
if TORCH_OK:
    x_10   = torch.tensor(np.append(features_9, xgb_score), dtype=torch.float32).unsqueeze(0)
    e_idx  = torch.tensor([[0], [0]], dtype=torch.long)
    e_attr = torch.tensor([[1.0]], dtype=torch.float32)
    with torch.no_grad():
        spatial_risk = float(np.clip(
            model(Data(x=x_10, edge_index=e_idx, edge_attr=e_attr)).item(), 0, 1))
    print(f"  Spatial risk : {spatial_risk:.3f}  (delta: {spatial_risk - xgb_score:+.3f})")
else:
    spatial_risk = xgb_score
    print(f"  GNN not loaded — using XGBoost: {spatial_risk:.3f}")

print("\nSTAGE 4 — Festival Predictor")
forecast = compute_stockout_forecast(
    current_stock=RAMESH['current_stock'],
    daily_sales=RAMESH['daily_sales'],
    festival_ctx=festival_ctx,
    lead_time_days=RAMESH['lead_time_days'],
)
print(f"  Effective demand/day : {forecast['effective_daily_demand']:.1f} units")
print(f"  Days until stockout  : {forecast['days_until_stockout']:.1f}")
print(f"  Recommended order    : {forecast['recommended_order_qty']} units")
print(f"  Urgency              : {forecast['restock_urgency']}")

print("\nSTAGE 5 — Distributor Recommendation")
DIST_INFO = [
    {"name": "FastTrack Logistics", "cost": 100, "eta": "4 hrs",  "rel": 0.99},
    {"name": "GraminRoute Hub",     "cost": 75,  "eta": "12 hrs", "rel": 0.95},
    {"name": "Budget Movers",       "cost": 60,  "eta": "24 hrs", "rel": 0.85},
]
rec_feats = np.array([[
    spatial_risk,
    min(forecast['days_until_stockout'] / 30.0, 1.0),
    min(festival_ctx['days_to_festival']  / 30.0, 1.0),
    RAMESH['credit_score'] / 900.0,
    min(forecast['recommended_order_qty'] / 100.0, 1.0),
]])
rec_proba = xgb_rec.predict_proba(rec_feats)[0]
ranked = sorted(
    [{"name": d["name"], "conf": round(float(rec_proba[i]), 3),
      "cost": d["cost"], "eta": d["eta"]} for i, d in enumerate(DIST_INFO)],
    key=lambda x: x["conf"], reverse=True)

medals = ['1st', '2nd', '3rd']
for i, r in enumerate(ranked):
    print(f"  {medals[i]}  {r['name']:<22} conf={r['conf']:.1%}  Rs{r['cost']}  {r['eta']}")
print(f"\n  TOP PICK: {ranked[0]['name']}")


## Training Complete

All models saved to `backend/models/`:

| File | Model |
|------|-------|
| `xgb_risk_model.pkl` | XGBoost Risk (9 features) |
| `xgb_recommender.pkl` | XGBoost Recommender (3 distributors) |
| `spatial_gnn.pth` | SpatialGNN (GATv2, in_dim=10) |

```bash
cd backend && uvicorn api.main:app --reload
# http://localhost:8000/docs
```
